In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
file_path = "/Volumes/workspace/default/kaggle_datasets/employee_salary_dataset.csv"

# emp_df = spark.read.csv(file_path, header=True, inferSchema=True)
emp_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file_path)
display(emp_df)

In [0]:
emp_df_stats = (
    emp_df
    .groupBy("Gender")
    .agg(F.count("*").alias("count"))
)
emp_df_stats.display()

In [0]:
emp_df.createOrReplaceTempView("employee")

In [0]:
%sql

select
City,
Gender,
case 
  when Gender = 'Male' then 1 else 0
end as Male,
case
  when Gender = 'Female' then 1 else 0
end as Female
from employee;

In [0]:
%sql
select
City,
sum(
  case 
    when Gender = 'Male' then 1 else 0
  end
) as Male,
sum(
  case
    when Gender = 'Female' then 1 else 0
  end
) as Female
from employee
group by City;

In [0]:
emp_stats_df = (
    emp_df
    .groupBy("City")
    .agg(
        F.sum(F.when(F.col("Gender") == "Male", 1).otherwise(0)).alias("MaleCount"),
        F.sum(F.when(F.col("Gender") == "Female", 1).otherwise(0)).alias("FemaleCount")
    )
)
emp_stats_df.display()

In [0]:
emp_df.count()

In [0]:
emp_df.columns

In [0]:
%sql
select 
EmployeeID,
Name,
Department,
Monthly_Salary
from
(
  select
    EmployeeID,
    Name,
    Department,
    Monthly_Salary,
    row_number() over(partition by Department order by Monthly_Salary desc) as rn
    from employee
) as e
where e.rn = 1;

In [0]:
%sql
select
EmployeeID,
Name,
Department,
Monthly_Salary
from employee
qualify row_number() over(partition by Department order by Monthly_Salary desc) = 1;

In [0]:
windowSpec = Window.partitionBy("Department").orderBy(F.col("Monthly_Salary").desc())

emp_df = (
    emp_df
    .withColumn("rn", F.row_number().over(windowSpec))
    .filter(F.col("rn") == 1)
    .select(
        "EmployeeID",
        "Name",
        "Department",
        "Monthly_Salary"
    )

)
emp_df.display()